# RVC — Chỉ chạy **Bước 4** (index FAISS)

Dùng khi bạn **đã interrupt** notebook train (Bước 3) nhưng vẫn muốn tạo file **`added_*.index`** để infer có retrieval.

## Điều kiện (quan trọng)

- **Có `logs/<tên_thí_nghiệm>/3_feature768/`** (hoặc `3_feature256` nếu v1) với nhiều file `.npy` — tức là **Bước 2** (F0 + Hubert) **đã chạy xong**.
- Bước 4 **không đọc** `G_*.pth` / `D_*.pth`; chỉ gom vector trong thư mục feature. Bạn có checkpoint generator vẫn **chạy index được** miễn có feature.
- **Working directory** = thư mục **`rvc_standalone`** (cùng mức với `infer/`, `training_pipeline/`).

## Kết quả

- File infer cần: **`added_IVF..._Flat_nprobe_..._<tên>_v2.index`** trong `logs/<tên_thí_nghiệm>/`.
- Đừng chọn nhầm file `trained_*.index` trên WebUI — dùng **`added_*.index`**.

## Bước A — Bootstrap (giống notebook train)

Nạp `Config`, `sys.path`, `cwd` về gốc `rvc_standalone`. Chạy **một lần** sau khi mở notebook / restart kernel.

In [2]:
import logging
import os
import pathlib
import sys

STANDALONE_ROOT = pathlib.Path.cwd().resolve()
if not (STANDALONE_ROOT / "infer" / "modules" / "train" / "train.py").is_file():
    raise SystemExit(
        "cwd phải là rvc_standalone. File → Open Folder → chọn rvc_standalone, rồi mở notebook này."
    )

os.chdir(STANDALONE_ROOT)
if str(STANDALONE_ROOT) not in sys.path:
    sys.path.insert(0, str(STANDALONE_ROOT))

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

from training_pipeline.setup_env import bootstrap
from training_pipeline.params import TrainingParams
from training_pipeline import steps as train_steps

root, config = bootstrap()
assert root == STANDALONE_ROOT

print("Gốc:", STANDALONE_ROOT)
print("=== Bootstrap xong ===")

INFO | Found GPU NVIDIA GeForce RTX 3050 Laptop GPU
INFO | Half-precision floating-point: True, device: cuda:0


Gốc: D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone
=== Bootstrap xong ===


## Bước B — Tham số (**phải trùng** lúc train)

- `experiment_name`: phải **đúng** thư mục `logs/giong_A` bạn đang dùng (ví dụ `giong_A`).
- `version`: **`v2`** → index dimension 768; **`v1`** → 256 (phải khớp lúc extract feature).

Chỉnh xong rồi chạy ô kiểm tra + ô index bên dưới.

In [3]:
from pathlib import Path

p = TrainingParams(
    experiment_name="giong_A",
    trainset_dir="datasets/giong_cua_toi",
    sample_rate_label="40k",
    version="v2",
    if_f0=True,
)

fea = (
    STANDALONE_ROOT / "logs" / p.experiment_name / "3_feature768"
    if p.version == "v2"
    else STANDALONE_ROOT / "logs" / p.experiment_name / "3_feature256"
)
npys = list(fea.glob("*.npy")) if fea.is_dir() else []
print("Thư mục feature:", fea)
print("Số file .npy:", len(npys))
if not npys:
    raise SystemExit(
        "Không có feature .npy — không chạy được Bước 4. Cần chạy lại Bước 2 trong notebook train."
    )
print("OK — có thể chạy Bước 4 (index)")

Thư mục feature: D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone\logs\giong_A\3_feature768
Số file .npy: 89
OK — có thể chạy Bước 4 (index)


## Bước 4 — Tạo index FAISS

Có thể **mất vài phút** tùy số vector. Output in ra tên file **`added_...index`**.

In [4]:
for line in train_steps.step_train_index(STANDALONE_ROOT, config, p):
    print(line)
print("=== Bước 4 xong ===")

INFO | Loading faiss with AVX2 support.
INFO | Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
INFO | Loading faiss.
INFO | Successfully loaded faiss.


(12746, 768),326
(12746, 768),326
training
(12746, 768),326
training
adding
(12746, 768),326
training
adding
成功构建索引 added_IVF326_Flat_nprobe_1_giong_A_v2.index
链接/复制索引失败: 'D:\\DUT_ITF\\Semester_10th\\do_an_tot_nghiep\\example_training_voice\\Retrieval-based-Voice-Conversion-WebUI\\rvc_standalone\\logs\\giong_A\\added_IVF326_Flat_nprobe_1_giong_A_v2.index' and 'assets\\indices\\giong_A_added_IVF326_Flat_nprobe_1_giong_A_v2.index' are the same file
=== Bước 4 xong ===


## (Tùy chọn) Xuất `.pth` nhỏ để infer

Nếu cần file generator gọn cho WebUI, chạy ô sau (dùng `G_2333333.pth` hoặc checkpoint mới nhất trong `logs/<tên>/`).

In [5]:
p.infer_weight_name = "giong_A_infer"
p.g_checkpoint_for_extract = ""  # để trống = ưu tiên G_2333333.pth
print(train_steps.step_extract_small_weights(STANDALONE_ROOT, p))

Success.
